# Cox Feature Selection - Optimized, with MLflow tracking

Demonstrates the fix for the long-running feature-selection problem in notebook
`the original survival pipeline` ("long hours running the model"), and uses **MLflow**
(built into Microsoft Fabric) to track the experiment.

**Pipeline:** seed a Lakehouse Delta table -> fetch it -> run the **same** best-
candidate forward selection two ways (**naive** = re-extract + re-encode every
trial; **cached** = extract once / encode once) -> log everything to MLflow.

Each step below has a short explainer and an **Area of improvement** callout.

### 1. Install dependencies and imports

**What it does:** installs `lifelines` into the running session and imports
pandas / numpy / Spark functions and **MLflow** (already wired to this Fabric
workspace, no setup needed).

> **Area of improvement** - the inline install runs on every cold session. For
> production, bake `lifelines` into a **Fabric Environment** so sessions start
> ready and reproducible instead of installing at runtime.

In [ ]:
import time
import hashlib
import subprocess
import sys
import os
from concurrent.futures import ThreadPoolExecutor

# Job-safe install: installs into the running driver env WITHOUT a %pip session restart
try:
    from lifelines import CoxPHFitter
    from lifelines.utils import concordance_index
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "lifelines"])
    from lifelines import CoxPHFitter
    from lifelines.utils import concordance_index

import numpy as np
import pandas as pd
import pyspark.sql.functions as F
import mlflow

MAX_LEVELS_PER_CAT   = 50
MIN_CAT_COUNT        = 10
NEAR_ZERO_VAR_THRESH = 1e-8
PENALIZER_PRIMARY    = 0.1
PENALIZER_FALLBACK   = 0.5
L1_RATIO             = 0.0

### 2. Parameters

**What it does:** every knob for the run - table names, the Cox survival schema
columns, selection thresholds, and the **MLflow experiment name**. Tagged as a
Fabric *parameters* cell so it can be overridden at schedule/run time.

> **Area of improvement** - these mirror the original notebook's governance
> config. Externalize them (a config table or workspace variables) so analysts
> can change behavior without editing code.

In [ ]:
# PARAMETERS (override at run / schedule time)
LAKEHOUSE_TABLE      = "cox_attrition_demo"            # Delta table in the attached Lakehouse
RESULT_TABLE         = "cox_selected_features_demo"    # "" to skip writing results
MLFLOW_EXPERIMENT    = "cox_feature_selection_demo"    # MLflow experiment name (Fabric)
FORCE_RELOAD         = False                            # True to rebuild the table from source
PUBLIC_CSV_URL       = "https://raw.githubusercontent.com/PacktPublishing/Statistics-for-Machine-Learning/41e73f42da97c164859641c2add6e487cbc77402/Chapter04/WA_Fn-UseC_-HR-Employee-Attrition.csv"
UPSAMPLE_ROWS        = 0                                 # 0 = native 1,470 rows; e.g. 200000 to simulate scale

COL_ID               = "PERSON_KEY"
COL_DURATION         = "SURVIVAL_TIME_DAYS"
COL_EVENT            = "event"
COL_FACILITY         = "FACILITY_NAME"

FEATURE_COLUMNS      = []                                # [] = auto-detect all non-reserved columns
MAX_FEATURES         = 12
MIN_DELTA_ACCEPT     = 0.001
MAX_PANDAS_ROWS      = 200000
TEST_SIZE            = 0.25
RANDOM_SEED          = 42
RUN_NAIVE_COMPARISON = True
RUN_PARALLEL         = True      # also run the cached engine with parallel per-round fits
RUN_SPARK_DISTRIBUTED = True     # also run the cached engine distributed across Spark executors
N_JOBS               = -1        # -1/0 = all driver cores; 1 = sequential

### 3. Seed the Lakehouse Delta table (idempotent)

**What it does:** if `cox_attrition_demo` does not exist, downloads the public
dataset, maps it onto the Cox schema (`PERSON_KEY` / `SURVIVAL_TIME_DAYS` /
`event` / `FACILITY_NAME`), and writes it as a managed Delta table. Skips if the
table is already present.

> **Area of improvement** - in production the source table comes from an upstream
> pipeline; this self-seed is only for a portable demo. A Fabric Spark pool may
> also block outbound internet, so prefer seeding from a trusted ingestion job.

In [ ]:
def _table_exists(name):
    try:
        spark.table(name)
        return True
    except Exception:
        return False

def seed_lakehouse_table():
    if _table_exists(LAKEHOUSE_TABLE) and not FORCE_RELOAD:
        print(f"[seed] {LAKEHOUSE_TABLE} already exists; skipping (set FORCE_RELOAD=True to rebuild)")
        return
    print(f"[seed] loading public dataset -> {LAKEHOUSE_TABLE}")
    pdf = pd.read_csv(PUBLIC_CSV_URL)

    pdf[COL_ID]       = pdf["EmployeeNumber"].astype(str)
    dur               = pd.to_numeric(pdf["YearsAtCompany"], errors="coerce") * 365.25
    pdf[COL_DURATION] = dur.where(dur > 0, 1.0)
    pdf[COL_EVENT]    = (pdf["Attrition"].astype(str).str.strip() == "Yes").astype(int)
    pdf[COL_FACILITY] = pdf["Department"].astype(str)

    drop_cols = ["EmployeeNumber", "Attrition", "YearsAtCompany", "Department",
                 "EmployeeCount", "Over18", "StandardHours"]
    pdf = pdf.drop(columns=[c for c in drop_cols if c in pdf.columns])

    if UPSAMPLE_ROWS and UPSAMPLE_ROWS > len(pdf):
        rng = np.random.default_rng(RANDOM_SEED)
        idx = rng.integers(0, len(pdf), size=UPSAMPLE_ROWS)
        pdf = pdf.iloc[idx].reset_index(drop=True)
        pdf[COL_ID] = (pdf.index + 1).astype(str)

    sdf = spark.createDataFrame(pdf)
    (sdf.write.mode("overwrite").option("overwriteSchema", "true")
        .format("delta").saveAsTable(LAKEHOUSE_TABLE))
    print(f"[seed] wrote {sdf.count():,} rows and {len(sdf.columns)} columns to {LAKEHOUSE_TABLE}")

seed_lakehouse_table()

### 4. Fetch from the Lakehouse

**What it does:** reads the Delta table with Spark and auto-detects the candidate
feature columns (everything that is not an id / target / strata column).

> **Area of improvement** - select only the columns you need and push filters
> down so you never pull all ~1,300 columns into the driver.

In [ ]:
sdf_source = spark.read.table(LAKEHOUSE_TABLE)
print(f"fetched {sdf_source.count():,} rows, {len(sdf_source.columns)} columns from {LAKEHOUSE_TABLE}")
display(sdf_source.limit(5))

reserved = {COL_ID, COL_DURATION, COL_EVENT, COL_FACILITY}
candidate_features = (
    [c for c in FEATURE_COLUMNS if c in sdf_source.columns and c not in reserved]
    if FEATURE_COLUMNS else
    [c for c in sdf_source.columns if c not in reserved]
)
print(f"{len(candidate_features)} candidate features:", candidate_features)

### 5. Core helpers - the optimization lives here

**What it does:** `build_trial_pdf_from_spark` pulls data to pandas;
`encode_for_cox` builds the one-hot design (impute, rare-level collapse,
`drop_first`, low-variance drop); `_fit_cox` fits a penalized, facility-stratified
Cox model; `score_design` returns **both train and validation C-index** (the
survival analog of ROC-AUC).

> **Area of improvement (THE FIX)** - in the original notebook
> `build_trial_pdf_from_spark` (a Spark `.count()` + `.toPandas()`) and
> `encode_for_cox` are re-run on **every candidate fit**. That repeated
> extraction + encoding is the "long hours." These helpers are written so the
> next cell can do it **once per scope** and reuse the result.

In [ ]:
def _is_test_series(keys):
    thr = int(TEST_SIZE * 1000)
    return keys.astype(str).map(
        lambda k: (int(hashlib.md5(k.encode()).hexdigest(), 16) % 1000) < thr
    )

def build_trial_pdf_from_spark(sdf, feature_list, sample_cap=None):
    feats = [c for i, c in enumerate(feature_list) if c in sdf.columns and c not in feature_list[:i]]
    cols  = [COL_ID, COL_EVENT, COL_DURATION, COL_FACILITY] + feats
    cols  = [c for i, c in enumerate(cols) if c in sdf.columns and c not in cols[:i]]
    s = (sdf.select(*cols)
            .withColumn(COL_EVENT, F.col(COL_EVENT).cast("int"))
            .withColumn(COL_DURATION, F.col(COL_DURATION).cast("double"))
            .withColumn(COL_FACILITY, F.trim(F.col(COL_FACILITY).cast("string")))
            .filter(F.col(COL_EVENT).isin(0, 1))
            .filter(F.col(COL_DURATION).isNotNull())
            .filter(F.col(COL_FACILITY).isNotNull()))
    s = s.withColumn(COL_DURATION,
                     F.when(F.col(COL_DURATION) < 1, F.lit(1.0)).otherwise(F.col(COL_DURATION)))
    cap = sample_cap if sample_cap is not None else MAX_PANDAS_ROWS
    total = s.count()                       # Spark action -- paid on EVERY naive call
    if total > cap:
        s = s.sample(False, min(float(cap) / float(total), 1.0), seed=RANDOM_SEED).limit(cap)
    pdf = s.toPandas()                       # Spark collect -- paid on EVERY naive call
    pdf[COL_EVENT]    = pd.to_numeric(pdf[COL_EVENT], errors="coerce").fillna(0).astype(int)
    pdf[COL_DURATION] = pd.to_numeric(pdf[COL_DURATION], errors="coerce")
    pdf = pdf.dropna(subset=[COL_DURATION, COL_EVENT, COL_FACILITY])
    pdf = pdf[pdf[COL_DURATION] >= 1].copy()
    pdf[COL_FACILITY] = pdf[COL_FACILITY].astype("object").fillna("UNKNOWN")
    pdf["_IS_TEST"] = _is_test_series(pdf[COL_ID])
    return pdf

def encode_for_cox(base_pdf, feature_list):
    feats = [c for i, c in enumerate(feature_list) if c not in feature_list[:i]]
    keep = [COL_DURATION, COL_EVENT, COL_FACILITY, "_IS_TEST"] + feats
    keep = [c for i, c in enumerate(keep) if c in base_pdf.columns and c not in keep[:i]]
    source = base_pdf[keep].copy()

    numeric_cols, categorical_cols = [], []
    for name in feats:
        if name not in source.columns:
            continue
        if pd.api.types.is_numeric_dtype(source[name]):
            numeric_cols.append(name)
        elif source[name].nunique(dropna=True) <= MAX_LEVELS_PER_CAT:
            categorical_cols.append(name)

    reserved2 = {COL_DURATION, COL_EVENT, COL_FACILITY, "_IS_TEST"}
    encoded = source[[COL_DURATION, COL_EVENT, COL_FACILITY, "_IS_TEST"] + numeric_cols + categorical_cols].copy()

    for name in numeric_cols:
        vals = pd.to_numeric(encoded[name], errors="coerce").replace([np.inf, -np.inf], np.nan)
        med = vals.median()
        encoded[name] = vals.fillna(0.0 if pd.isna(med) else med)

    for name in categorical_cols:
        vals = encoded[name].astype("object").fillna("MISSING").astype(str).str.strip().replace("", "MISSING")
        counts = vals.value_counts(dropna=False)
        rare = counts[counts < MIN_CAT_COUNT].index
        encoded[name] = vals.where(~vals.isin(rare), other="OTHER")

    if categorical_cols:
        encoded = pd.get_dummies(encoded, columns=categorical_cols, prefix=categorical_cols,
                                 prefix_sep="__LEVEL__", drop_first=True)

    # convert dummy booleans to int, but keep reserved columns (e.g. _IS_TEST) untouched
    for c in encoded.columns:
        if c not in reserved2 and encoded[c].dtype == bool:
            encoded[c] = encoded[c].astype(int)

    feat_cols = [c for c in encoded.columns if c not in reserved2]
    if feat_cols:
        encoded[feat_cols] = encoded[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        variances = encoded[feat_cols].var(axis=0)
        low = variances[(variances <= 0) | (variances < NEAR_ZERO_VAR_THRESH)].index.tolist()
        if low:
            encoded = encoded.drop(columns=low, errors="ignore")

    provenance = {}
    for col in encoded.columns:
        if col in reserved2:
            continue
        src = col.split("__LEVEL__")[0] if "__LEVEL__" in col else col
        provenance.setdefault(src, []).append(col)
    return encoded, provenance

def _fit_cox(train_df):
    for pen in (PENALIZER_PRIMARY, PENALIZER_FALLBACK):
        try:
            m = CoxPHFitter(penalizer=pen, l1_ratio=L1_RATIO)
            m.fit(train_df, duration_col=COL_DURATION, event_col=COL_EVENT,
                  strata=[COL_FACILITY], show_progress=False)
            return m
        except Exception:
            continue
    return None

def score_design(encoded, feat_cols):
    """Return (train_c_index, validation_c_index) for a Cox fit on these columns."""
    if not feat_cols:
        return (float("-inf"), float("-inf"))
    design = encoded[[COL_DURATION, COL_EVENT, COL_FACILITY, "_IS_TEST"] + list(feat_cols)]
    is_test = design["_IS_TEST"].astype(bool)
    train_df = design[~is_test].drop(columns=["_IS_TEST"])
    test_df  = design[is_test].drop(columns=["_IS_TEST"])
    if train_df.empty or test_df.empty:
        return (float("-inf"), float("-inf"))
    for sp in (train_df, test_df):
        ev = int(sp[COL_EVENT].sum())
        if ev == 0 or ev == sp.shape[0]:
            return (float("-inf"), float("-inf"))
    m = _fit_cox(train_df)
    if m is None:
        return (float("-inf"), float("-inf"))
    train_scores = m.predict_partial_hazard(train_df)
    test_scores  = m.predict_partial_hazard(test_df)
    c_train = concordance_index(train_df[COL_DURATION], -train_scores, train_df[COL_EVENT])
    c_test  = concordance_index(test_df[COL_DURATION], -test_scores, test_df[COL_EVENT])
    return (float(c_train), float(c_test))

### 6. Forward selection - naive vs cached

**What it does:** TRUE best-candidate forward selection. Each round it fits every
remaining candidate against the current set and admits the single best if its
**validation** C-index gain clears `MIN_DELTA_ACCEPT`. Both engines record a
per-round history of train/validation C-index so we can chart learning in MLflow.
- `forward_select_naive` - re-extracts + re-encodes for every candidate (current behavior).
- `forward_select_cached` - extracts once / encodes once, then slices the design per trial (the fix).
- `forward_select_cached_parallel` - same as cached, but the independent per-candidate Cox fits in each round run **concurrently** on the driver's cores (thread pool), so a round finishes in about `max` time instead of `sum` time.
- `forward_select_cached_spark` - distributes those same independent fits across Spark **executors** (whole cluster). lifelines stays single-node: each executor fits its OWN complete model on a broadcast copy of the encoded frame; Spark is only the scheduler. Requires lifelines on the executors (Fabric Environment, with a runtime-install fallback for the demo).

> **Area of improvement** - the per-candidate fits within a round are independent,
> so they parallelize cleanly. Driver threads win on small/medium data (no Spark
> job overhead); Spark-executor distribution wins at scale (large data / costly
> fits / many candidates). A univariate pre-screen further shrinks the pool.

In [ ]:
def forward_select_naive(sdf, candidates):
    stats = {"fits": 0, "extract_encode_s": 0.0, "fit_s": 0.0}
    selected, current_c, history = [], 0.0, []
    remaining = list(candidates)
    while remaining and len(selected) < MAX_FEATURES:
        best = None  # (feature, c_train, c_test)
        for cand in remaining:
            trial = selected + [cand]
            t0 = time.perf_counter()
            pdf = build_trial_pdf_from_spark(sdf, trial)      # re-extract from Spark
            enc, prov = encode_for_cox(pdf, trial)            # re-encode
            stats["extract_encode_s"] += time.perf_counter() - t0
            feat_cols = [c for f in trial for c in prov.get(f, [])]
            t1 = time.perf_counter()
            c_train, c_test = score_design(enc, feat_cols)
            stats["fit_s"] += time.perf_counter() - t1
            stats["fits"] += 1
            if best is None or c_test > best[2]:
                best = (cand, c_train, c_test)
        if best is None or (best[2] - current_c) < MIN_DELTA_ACCEPT:
            break
        selected.append(best[0]); remaining.remove(best[0]); current_c = best[2]
        history.append({"round": len(selected), "feature": best[0], "c_train": best[1], "c_test": best[2]})
    return selected, history, stats

def forward_select_cached(sdf, candidates):
    stats = {"fits": 0, "extract_encode_s": 0.0, "fit_s": 0.0}
    t0 = time.perf_counter()
    base = build_trial_pdf_from_spark(sdf, candidates)        # extract ONCE
    enc, prov = encode_for_cox(base, candidates)              # encode ONCE
    stats["extract_encode_s"] += time.perf_counter() - t0
    selected, current_c, history = [], 0.0, []
    remaining = [c for c in candidates if c in prov]
    while remaining and len(selected) < MAX_FEATURES:
        sel_cols = [c for f in selected for c in prov.get(f, [])]
        best = None  # (feature, c_train, c_test)
        for cand in remaining:
            feat_cols = sel_cols + prov.get(cand, [])
            t1 = time.perf_counter()
            c_train, c_test = score_design(enc, feat_cols)
            stats["fit_s"] += time.perf_counter() - t1
            stats["fits"] += 1
            if best is None or c_test > best[2]:
                best = (cand, c_train, c_test)
        if best is None or (best[2] - current_c) < MIN_DELTA_ACCEPT:
            break
        selected.append(best[0]); remaining.remove(best[0]); current_c = best[2]
        history.append({"round": len(selected), "feature": best[0], "c_train": best[1], "c_test": best[2]})
    return selected, history, stats

def forward_select_cached_parallel(sdf, candidates, n_jobs=-1):
    # Identical selection to forward_select_cached, but the per-candidate fits in
    # each round are submitted to a thread pool and run concurrently.
    stats = {"fits": 0, "extract_encode_s": 0.0, "fit_s": 0.0}
    t0 = time.perf_counter()
    base = build_trial_pdf_from_spark(sdf, candidates)        # extract ONCE
    enc, prov = encode_for_cox(base, candidates)              # encode ONCE
    stats["extract_encode_s"] += time.perf_counter() - t0
    workers = (os.cpu_count() or 1) if (n_jobs is None or n_jobs <= 0) else int(n_jobs)
    selected, current_c, history = [], 0.0, []
    remaining = [c for c in candidates if c in prov]
    with ThreadPoolExecutor(max_workers=workers) as pool:
        while remaining and len(selected) < MAX_FEATURES:
            sel_cols = [c for f in selected for c in prov.get(f, [])]
            def _trial(cand):
                c_train, c_test = score_design(enc, sel_cols + prov.get(cand, []))
                return (cand, c_train, c_test)
            t1 = time.perf_counter()
            results = list(pool.map(_trial, remaining))       # fits run concurrently on driver cores
            stats["fit_s"] += time.perf_counter() - t1
            stats["fits"] += len(remaining)
            best = None  # input order preserved -> deterministic tie-break, same as sequential
            for cand, c_train, c_test in results:
                if best is None or c_test > best[2]:
                    best = (cand, c_train, c_test)
            if best is None or (best[2] - current_c) < MIN_DELTA_ACCEPT:
                break
            selected.append(best[0]); remaining.remove(best[0]); current_c = best[2]
            history.append({"round": len(selected), "feature": best[0], "c_train": best[1], "c_test": best[2]})
    return selected, history, stats

def _ensure_lifelines_on_executor():
    # Runs ON each executor. lifelines should be provided by a Fabric Environment;
    # this best-effort install keeps the demo working without one.
    try:
        from lifelines import CoxPHFitter
        from lifelines.utils import concordance_index
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "lifelines"])
        from lifelines import CoxPHFitter
        from lifelines.utils import concordance_index
    return CoxPHFitter, concordance_index

def forward_select_cached_spark(sdf, candidates):
    # Distributes the independent per-candidate fits across Spark EXECUTORS.
    # lifelines stays single-node: each executor fits its OWN complete model on a
    # broadcast copy of the (small) encoded pandas frame -- Spark is the scheduler.
    stats = {"fits": 0, "extract_encode_s": 0.0, "fit_s": 0.0}
    t0 = time.perf_counter()
    base = build_trial_pdf_from_spark(sdf, candidates)        # extract ONCE (driver)
    enc, prov = encode_for_cox(base, candidates)              # encode ONCE (driver)
    stats["extract_encode_s"] += time.perf_counter() - t0

    sc = spark.sparkContext
    bc_enc = sc.broadcast(enc)                                # ship encoded frame to executors once
    col_dur, col_evt, col_fac = COL_DURATION, COL_EVENT, COL_FACILITY
    pen1, pen2, l1 = PENALIZER_PRIMARY, PENALIZER_FALLBACK, L1_RATIO

    def _fit_on_executor(item):
        cand, feat_cols = item
        CoxPHFitter, concordance_index = _ensure_lifelines_on_executor()
        if not feat_cols:
            return (cand, float("-inf"), float("-inf"))
        enc_local = bc_enc.value
        design = enc_local[[col_dur, col_evt, col_fac, "_IS_TEST"] + list(feat_cols)]
        is_test = design["_IS_TEST"].astype(bool)
        tr = design[~is_test].drop(columns=["_IS_TEST"])
        te = design[is_test].drop(columns=["_IS_TEST"])
        if tr.empty or te.empty:
            return (cand, float("-inf"), float("-inf"))
        for part in (tr, te):
            ev = int(part[col_evt].sum())
            if ev == 0 or ev == part.shape[0]:
                return (cand, float("-inf"), float("-inf"))
        model = None
        for pen in (pen1, pen2):
            try:
                m = CoxPHFitter(penalizer=pen, l1_ratio=l1)
                m.fit(tr, duration_col=col_dur, event_col=col_evt, strata=[col_fac], show_progress=False)
                model = m
                break
            except Exception:
                continue
        if model is None:
            return (cand, float("-inf"), float("-inf"))
        ctr = concordance_index(tr[col_dur], -model.predict_partial_hazard(tr), tr[col_evt])
        cte = concordance_index(te[col_dur], -model.predict_partial_hazard(te), te[col_evt])
        return (cand, float(ctr), float(cte))

    selected, current_c, history = [], 0.0, []
    remaining = [c for c in candidates if c in prov]
    while remaining and len(selected) < MAX_FEATURES:
        sel_cols = [c for f in selected for c in prov.get(f, [])]
        payload = [(cand, sel_cols + prov.get(cand, [])) for cand in remaining]
        t1 = time.perf_counter()
        results = sc.parallelize(payload, numSlices=len(payload)).map(_fit_on_executor).collect()
        stats["fit_s"] += time.perf_counter() - t1
        stats["fits"] += len(remaining)
        order = {cand: i for i, cand in enumerate(remaining)}
        results.sort(key=lambda r: order[r[0]])              # deterministic tie-break, same as sequential
        best = None
        for cand, c_train, c_test in results:
            if best is None or c_test > best[2]:
                best = (cand, c_train, c_test)
        if best is None or (best[2] - current_c) < MIN_DELTA_ACCEPT:
            break
        selected.append(best[0]); remaining.remove(best[0]); current_c = best[2]
        history.append({"round": len(selected), "feature": best[0], "c_train": best[1], "c_test": best[2]})
    bc_enc.unpersist()
    return selected, history, stats

### 7. Run both engines and track everything in MLflow

**What it does:** runs the cached (and optionally naive) engine inside **MLflow
runs** logged to this Fabric workspace. A parent run holds the comparison; nested
`cached` / `cached_parallel` / `cached_spark` / `naive` runs each log:
- **parameters** - engine, max features, delta threshold, candidate count, penalizer, selected features;
- **metrics** - per-round `train_c_index` and `val_c_index` (stepped), final scores, number of Cox fits, extract+encode seconds, fit seconds, total seconds;
- the **speedups** on the parent - `speedup_x` (cached vs naive) and `parallel_speedup_vs_sequential_x` (parallel vs sequential), plus extract+encode / fit seconds saved;
- an **artifact table** of the selected features with their train/validation C-index.

Open **Experiments -> cox_feature_selection_demo** in the workspace to compare runs.

> **Area of improvement** - register the final Cox model in the MLflow Model
> Registry and add continuous evaluation so C-index drift is tracked over time.

In [ ]:
mlflow.autolog(disable=True)                 # explicit logging only -- keeps the experiment clean
mlflow.set_experiment(MLFLOW_EXPERIMENT)

def _log_feature_table(history, artifact_file):
    tbl = {
        "rank":          [h["round"]   for h in history],
        "feature":       [h["feature"] for h in history],
        "train_c_index": [round(h["c_train"], 6) for h in history],
        "val_c_index":   [round(h["c_test"], 6)  for h in history],
    }
    try:
        mlflow.log_table(data=tbl, artifact_file=artifact_file)
    except Exception:
        mlflow.log_dict(tbl, artifact_file)

def run_and_log(engine_name, fn):
    with mlflow.start_run(run_name=engine_name, nested=True):
        mlflow.set_tag("engine", engine_name)
        t0 = time.perf_counter()
        selected, history, stats = fn(sdf_source, candidate_features)
        wall = time.perf_counter() - t0

        mlflow.log_params({
            "engine": engine_name,
            "max_features": MAX_FEATURES,
            "min_delta_accept": MIN_DELTA_ACCEPT,
            "n_candidates": len(candidate_features),
            "penalizer": PENALIZER_PRIMARY,
            "test_size": TEST_SIZE,
            "n_selected": len(selected),
            "selected_features": ", ".join(selected) if selected else "(none)",
        })
        for h in history:
            mlflow.log_metric("train_c_index", h["c_train"], step=h["round"])
            mlflow.log_metric("val_c_index",   h["c_test"],  step=h["round"])
        final_train = history[-1]["c_train"] if history else float("nan")
        final_val   = history[-1]["c_test"]  if history else float("nan")
        mlflow.log_metrics({
            "final_train_c_index": final_train,
            "final_val_c_index": final_val,
            "n_cox_fits": stats["fits"],
            "extract_encode_seconds": round(stats["extract_encode_s"], 3),
            "fit_seconds": round(stats["fit_s"], 3),
            "total_seconds": round(wall, 3),
        })
        # consistent names so these compare side-by-side with the AutoML/AFT/MLlib runs
        mlflow.log_metrics({"test_cindex": final_val, "seconds": round(wall, 3)})
        mlflow.log_params({"library": "lifelines", "model": "cox_ph",
                           "approach": "forward_selection", "compute": engine_name})
        if history:
            _log_feature_table(history, f"{engine_name}_selected_features.json")

        print(f"[{engine_name}] selected = {selected}")
        print(f"[{engine_name}] final val C-index={final_val:.6f} | fits={stats['fits']} | "
              f"extract+encode={stats['extract_encode_s']:.2f}s | fit={stats['fit_s']:.2f}s | total={wall:.2f}s")
        return selected, wall, stats, history

with mlflow.start_run(run_name="forward_selection_comparison"):
    mlflow.log_params({
        "dataset_table": LAKEHOUSE_TABLE,
        "n_candidates": len(candidate_features),
        "max_features": MAX_FEATURES,
    })

    sel_c, wall_c, st_c, hist_c = run_and_log("cached", forward_select_cached)
    selected_features = sel_c

    if RUN_PARALLEL:
        mlflow.log_param("n_jobs", N_JOBS)
        sel_p, wall_p, st_p, hist_p = run_and_log(
            "cached_parallel", lambda s, c: forward_select_cached_parallel(s, c, N_JOBS))
        mlflow.log_param("parallel_identical_to_cached", sel_p == sel_c)
        mlflow.log_metric("parallel_speedup_vs_sequential_x", round(wall_c / wall_p, 2) if wall_p > 0 else 0.0)
        mlflow.log_metric("parallel_fit_seconds_saved", round(st_c["fit_s"] - st_p["fit_s"], 3))
        print(f"\nPARALLEL identical_to_cached={sel_p == sel_c} | "
              f"fit {st_c['fit_s']:.2f}s -> {st_p['fit_s']:.2f}s | "
              f"total {wall_c:.2f}s -> {wall_p:.2f}s | speedup={wall_c / max(wall_p, 1e-9):.1f}x")

    if RUN_SPARK_DISTRIBUTED:
        try:
            sel_s, wall_s, st_s, hist_s = run_and_log("cached_spark", forward_select_cached_spark)
            mlflow.log_param("spark_identical_to_cached", sel_s == sel_c)
            mlflow.log_metric("spark_speedup_vs_sequential_x", round(wall_c / wall_s, 2) if wall_s > 0 else 0.0)
            print(f"\nSPARK-DISTRIBUTED identical_to_cached={sel_s == sel_c} | "
                  f"total seq={wall_c:.2f}s vs spark={wall_s:.2f}s | fits spread across executors")
        except Exception as e:
            mlflow.log_param("spark_distributed_status", f"skipped: {str(e)[:200]}")
            print("cached_spark skipped (needs lifelines on executors / Fabric Environment):", e)

    if RUN_NAIVE_COMPARISON:
        sel_n, wall_n, st_n, hist_n = run_and_log("naive", forward_select_naive)
        identical = (sel_n == sel_c)
        mlflow.log_param("identical_features", identical)
        mlflow.log_metric("speedup_x", round(wall_n / wall_c, 2) if wall_c > 0 else 0.0)
        mlflow.log_metric("extract_encode_saved_seconds",
                          round(st_n["extract_encode_s"] - st_c["extract_encode_s"], 3))
        print(f"\nPARITY identical_features={identical} | "
              f"SPEEDUP={wall_n / wall_c:.1f}x | "
              f"extract+encode {st_n['extract_encode_s']:.2f}s -> {st_c['extract_encode_s']:.2f}s")

    if hist_c:
        _log_feature_table(hist_c, "selected_features.json")

print(f"\nMLflow experiment: {MLFLOW_EXPERIMENT}  (open 'Experiments' in the workspace to compare runs)")

### 8. Persist results

**What it does:** writes the chosen features to `cox_selected_features_demo` and
displays them. (The MLflow run already captured the scored history as an
artifact.)

> **Area of improvement** - also emit a model-card / data-card artifact and the
> per-round metrics table for governance and audit.

In [ ]:
if RESULT_TABLE and selected_features:
    res = spark.createDataFrame(
        pd.DataFrame({"rank": list(range(1, len(selected_features) + 1)), "feature": selected_features})
    )
    (res.write.mode("overwrite").option("overwriteSchema", "true")
        .format("delta").saveAsTable(RESULT_TABLE))
    print("wrote", RESULT_TABLE)
    display(spark.read.table(RESULT_TABLE))

### 9. Register the model in MLflow, reload it, and score one record

**What it does:** fits the final Cox model on the selected features, logs it to
MLflow as a reusable **pyfunc** model, **loads it back** from the MLflow run, and
scores a **single** hold-out employee - printing the model's risk output next to
the actual label.

> **Area of improvement** - promote this logged model through the MLflow Model
> Registry (staging -> production) and serve it behind a REST endpoint for
> real-time scoring.

In [ ]:
import mlflow.pyfunc

selected_features = selected_features if "selected_features" in globals() else []
if not selected_features:
    print("No features were selected; skipping model logging.")
else:
    base_m = build_trial_pdf_from_spark(sdf_source, selected_features)
    enc_m, prov_m = encode_for_cox(base_m, selected_features)
    sel_cols_m = [c for f in selected_features for c in prov_m.get(f, [])]
    design_m = enc_m[[COL_DURATION, COL_EVENT, COL_FACILITY, "_IS_TEST"] + sel_cols_m]
    is_test_m = design_m["_IS_TEST"].astype(bool)
    train_m = design_m[~is_test_m].drop(columns=["_IS_TEST"])
    test_m  = design_m[is_test_m].drop(columns=["_IS_TEST"])
    final_model = _fit_cox(train_m)
    input_cols = [COL_FACILITY] + sel_cols_m

    if final_model is None or test_m.empty:
        print("Final Cox fit failed or no test rows; skipping model logging.")
    else:
        class CoxPHWrapper(mlflow.pyfunc.PythonModel):
            def __init__(self, model, input_cols):
                self.model = model
                self.input_cols = input_cols
            def predict(self, context, model_input):
                cols = [c for c in self.input_cols if c in model_input.columns]
                return self.model.predict_partial_hazard(model_input[cols])

        example = test_m[input_cols].head(1)
        with mlflow.start_run(run_name="final_cox_model") as model_run:
            mlflow.log_param("selected_features", ", ".join(selected_features))
            mlflow.pyfunc.log_model(
                artifact_path="cox_model",
                python_model=CoxPHWrapper(final_model, input_cols),
                input_example=example,
                pip_requirements=["lifelines", "pandas", "numpy", "scikit-learn"],
            )
            model_uri = f"runs:/{model_run.info.run_id}/cox_model"
        print("logged model:", model_uri)

        # reload the model from MLflow and score ONE hold-out record
        loaded_model = mlflow.pyfunc.load_model(model_uri)
        one = test_m.iloc[[0]]
        one_input = one[input_cols]
        pred = loaded_model.predict(one_input)
        risk = float(np.asarray(pred).ravel()[0])
        actual_event = int(one[COL_EVENT].iloc[0])
        actual_days = float(one[COL_DURATION].iloc[0])

        print("\n=== Single-record inference (model loaded from MLflow) ===")
        print("Input features:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in one_input.iloc[0].to_dict().items()})
        print(f"Model output  -> partial hazard (risk) = {risk:.4f}   (1.0 = baseline; >1 = higher attrition risk)")
        print(f"Actual label  -> event = {actual_event} (1 = left, 0 = stayed/censored) | tenure_days = {actual_days:.0f}")
        verdict = "elevated" if risk >= 1.0 else "lower"
        outcome = "LEFT" if actual_event == 1 else "STAYED/censored"
        print(f"Interpretation -> model predicts {verdict} risk; this employee actually {outcome}.")

        spark.createDataFrame(pd.DataFrame([{
            "partial_hazard_risk": round(risk, 4),
            "predicted_risk_level": verdict,
            "actual_event": actual_event,
            "actual_outcome": outcome,
            "tenure_days": round(actual_days, 0),
        }])).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("cox_single_prediction_demo")
        print("wrote cox_single_prediction_demo")